# BC v2 Training on Colab A100

Trains the ObjectCentricPolicy on the bc_v2 corpus (12,425 episodes, 4.32 GB gz) collected via `scripts/collect_gt_warmstart.py`.

**Plan:** see `.claude/doc/bc_v2_training_plan.md` (Path C, A100 cloud).

**Required runtime:** A100 40 GB. Notebook **aborts** if a different GPU is allocated (don't waste compute units on a T4).

**Required Drive upload (do this BEFORE starting the notebook):**
- The data file at `/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.jsonl.gz` (4.32 GB).

**Project code is cloned from GitHub** in Cell 4 — no Drive upload needed. Defaults to repo `https://github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git` branch `jihang`.

**Compute unit budget (200 units, A100 40 GB ≈ 11.77 units/h):**
- Smoke run (1 epoch on tiny subset): ~5–10 units
- Full bc_v2 training (16 epochs): ~100–110 units
- Reserve: ~80 units for one retry / iteration

Approx wall: **6–9 h continuous** for the full run. Colab session cap is 12 h, so a single uninterrupted session covers it. The training cell (§8) spawns a daemon thread that rsyncs `LOCAL_OUTPUT → DRIVE_OUTPUT` every 5 min, so a disconnect costs at most 5 min of progress.

**Notebook layout (9 sections):**
1. GPU verify (abort if not A100)
2. Drive mount
3. Path / run-name config (edit before running)
4. Clone project from GitHub
5. Copy data .gz from Drive to local SSD
6. Install bundled wheels
7. (optional) Smoke run
8. Full training (with inline 5-min Drive sync)
9. Resume after disconnect

## 1. Verify GPU is A100 (abort if not)

In [ ]:
import subprocess
import sys

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('Allocated GPU:', out)

if 'A100' not in out:
    raise SystemExit(
        f'ABORT: expected A100, got: {out!r}\n'
        f'Go to Runtime > Change runtime type and pick A100.\n'
        f'Do not run on a smaller GPU (profile a100 needs >12 GB VRAM and the wall on T4 / L4 will exceed your unit budget).'
    )
print('OK: A100 confirmed.')

## 2. Mount Google Drive

Required for reading the data .gz and writing checkpoints back so they survive session disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure paths and run name

Edit the values below to match your setup. Defaults clone the `jihang` branch.

In [ ]:
from pathlib import Path
from datetime import datetime

# ===== EDIT THESE =====
GITHUB_REPO_URL    = 'https://github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git'
GITHUB_BRANCH      = 'jihang'   # the user's working branch
DRIVE_INPUT_GZ     = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.jsonl.gz')
RUN_NAME           = 'bc_v2_baseline_a100'
EPOCHS_OVERRIDE    = None      # None -> use profile's epochs=16; else int
# ===== END EDIT =====

DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR     = Path('/content/ARC-Prize-2026-ARC-AGI-3')
LOCAL_INPUT_DIR   = Path('/content/bc_v2_input')
LOCAL_OUTPUT_DIR  = Path('/content/bc_v2_output')

RUN_TS         = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
FULL_RUN_NAME  = f'{RUN_NAME}_{RUN_TS}'
DRIVE_OUTPUT   = DRIVE_OUTPUT_BASE / FULL_RUN_NAME
LOCAL_OUTPUT   = LOCAL_OUTPUT_DIR / FULL_RUN_NAME

for p in [DRIVE_OUTPUT_BASE, LOCAL_INPUT_DIR, LOCAL_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

print('github repo      :', GITHUB_REPO_URL)
print('github branch    :', GITHUB_BRANCH)
print('drive input gz   :', DRIVE_INPUT_GZ, '(exists:', DRIVE_INPUT_GZ.exists(), ')')
print('drive output dir :', DRIVE_OUTPUT)
print('local workdir    :', LOCAL_WORKDIR)
print('local output dir :', LOCAL_OUTPUT)

if not DRIVE_INPUT_GZ.exists():
    raise SystemExit(f'Drive input .gz not found: {DRIVE_INPUT_GZ}. Upload the data first.')

## 4. Clone the project from GitHub (and switch to the working branch)

Clones into `/content/ARC-Prize-2026-ARC-AGI-3` and checks out `GITHUB_BRANCH` (default `jihang`).

If the directory already exists from a previous run, it's wiped and re-cloned to ensure a fresh, branch-correct copy.

**Private repo?** If `git clone` prompts for credentials or fails with 403/404, the repo is private. Two options:
1. Make the repo public, OR
2. Use a Personal Access Token: replace `GITHUB_REPO_URL` with `https://<USERNAME>:<TOKEN>@github.com/QAQWillQwQ/ARC-Prize-2026-ARC-AGI-3.git`.

In [ ]:
import shutil
import subprocess

if LOCAL_WORKDIR.exists():
    print(f'removing existing {LOCAL_WORKDIR} for a clean clone')
    shutil.rmtree(LOCAL_WORKDIR)

# Shallow clone: --depth 1 fetches only the latest commit on the branch.
# Saves time and disk vs full history; we don't need git log here.
print(f'git clone --depth 1 --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} {LOCAL_WORKDIR}')
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(LOCAL_WORKDIR)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(f'git clone failed (exit {result.returncode}). See stderr above.')

# Confirm we landed on the right branch + show the head commit
head = subprocess.run(['git', '-C', str(LOCAL_WORKDIR), 'log', '-1', '--format=%H %s'],
                      capture_output=True, text=True).stdout.strip()
branch = subprocess.run(['git', '-C', str(LOCAL_WORKDIR), 'rev-parse', '--abbrev-ref', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print(f'\nhead commit: {head}')
print(f'branch     : {branch}')
if branch != GITHUB_BRANCH:
    raise SystemExit(f'expected branch {GITHUB_BRANCH}, got {branch}')

# Verify the critical paths
print()
print('verify project layout:')
for sub in ['src', 'scripts', 'arc_agi_3_wheels', 'environment_files']:
    p = LOCAL_WORKDIR / sub
    print(f'  {sub}: {p.exists()} ({len(list(p.glob("*"))) if p.exists() else 0} entries)')

## 5. Copy training cache from Drive to local SSD

Reading directly from Drive is slow. Copying once front-loads the I/O cost.

**Use the prebuilt cache** (`episodes_bc_v2.cache.pkl.gz`, ~1 GB) — it's a numpy-uint8 cache of the corpus, **loads in ~2 min instead of 30+ min** for the raw `.gz` JSON parse. Build it locally once via `scripts/build_train_cache.py` and upload to Drive.

If only the raw gz is available, the cell falls back to it (training will be 30 min slower at the build pass).

In [ ]:
# --- Cell 5: copy training cache (or fallback gz) from Drive to local SSD --- #
import shutil

# Prefer cache; fall back to raw gz.
DRIVE_CACHE_CANDIDATES = [
    Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.cache.pkl.gz'),  # preferred
    Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data/episodes_bc_v2.jsonl.gz'),       # fallback
]
DRIVE_INPUT = next((p for p in DRIVE_CACHE_CANDIDATES if p.exists()), None)
if DRIVE_INPUT is None:
    raise SystemExit(
        f'No training data found on Drive. Tried:\n  ' +
        '\n  '.join(str(p) for p in DRIVE_CACHE_CANDIDATES)
    )

LOCAL_INPUT = LOCAL_INPUT_DIR / DRIVE_INPUT.name
if LOCAL_INPUT.exists() and LOCAL_INPUT.stat().st_size == DRIVE_INPUT.stat().st_size:
    print(f'already present: {LOCAL_INPUT} ({LOCAL_INPUT.stat().st_size / 1e9:.2f} GB)')
else:
    print(f'copying {DRIVE_INPUT} -> {LOCAL_INPUT} ...')
    shutil.copy2(DRIVE_INPUT, LOCAL_INPUT)
    print(f'done: {LOCAL_INPUT.stat().st_size / 1e9:.2f} GB')

print(f'using {"CACHE" if LOCAL_INPUT.name.endswith((".pkl.gz", ".pkl")) else "RAW GZ"}: {LOCAL_INPUT}')

## 6. Install bundled wheels

`arc_agi` and `arcengine` are not on PyPI for Linux. The wheels in `arc_agi_3_wheels/` are manylinux cp312 — they install cleanly on Colab Linux.

In [ ]:
wheels_dir = LOCAL_WORKDIR / 'arc_agi_3_wheels'
wheels = sorted(wheels_dir.glob('*.whl'))
print(f'installing {len(wheels)} wheels from {wheels_dir}')
!pip install -q {wheels_dir}/*.whl

!pip install -q py-spy

import importlib
for mod in ['arc_agi', 'arcengine', 'torch']:
    m = importlib.import_module(mod)
    print(f'  {mod}: {getattr(m, "__version__", "?")}  ({m.__file__})')

import torch
print(f'torch.cuda.is_available: {torch.cuda.is_available()}')
print(f'torch.cuda.get_device_name: {torch.cuda.get_device_name(0)}')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

## 7. (Optional) Smoke run: 1 epoch on a single game

Burns ~3–5 units to verify the training pipeline runs end-to-end before committing to the long run.

Skip this if you've already validated the pipeline. Set `RUN_SMOKE = False` to skip.

In [ ]:
RUN_SMOKE = False  # default OFF since the build pass on raw gz is slow; only flip if you suspect a pipeline issue

if RUN_SMOKE:
    smoke_out = LOCAL_OUTPUT_DIR / f'smoke_{RUN_TS}'
    smoke_out.mkdir(parents=True, exist_ok=True)
    print(f'smoke run -> {smoke_out}')
    !cd {LOCAL_WORKDIR} && python -u -m src.train \
        --project-root {LOCAL_WORKDIR} \
        --data {LOCAL_INPUT} \
        --output-dir {smoke_out} \
        --hardware-profile a100 \
        --games sp80 \
        --epochs 1 \
        --online-val-every 1000
    print('--- smoke files written ---')
    !ls -la {smoke_out}/checkpoints/ 2>/dev/null
else:
    print('skipping smoke (RUN_SMOKE=False) — go straight to §8')

## 8. Full training run (with inline 5-min Drive sync)

Uses the `a100` hardware profile (batch=192, model_dim=384, depth=6, slots=8, history=4, epochs=16, online_val_every=2).

A background daemon thread rsyncs `LOCAL_OUTPUT → DRIVE_OUTPUT` every 5 minutes throughout training, so a Colab disconnect costs at most 5 min of progress (last.pth + metrics.csv land on Drive). One Colab runtime, one unit cost — no parallel tab needed.

stdout is `tee`'d into `train.log` inside `LOCAL_OUTPUT`, so the same sync also makes the training log readable from Drive.

**Caveat:** this notebook uses the **existing** `train.py` flow. The `bc_v2_3stage` curriculum + bucket weighting + phase mask described in §4–§6 of `bc_v2_training_plan.md` are NOT yet implemented in train.py — this is a single-stage baseline across all buckets at equal weight.

**Wall estimate:** 6–9 h.

In [ ]:
import threading
import time
import subprocess

epochs_arg = f'--epochs {EPOCHS_OVERRIDE}' if EPOCHS_OVERRIDE else ''

# ===== Knobs =====
# MAX_TRANS_PER_EP=1000 caps in-RAM dataset to ~50 GB (vs ~124 GB without cap).
# BATCH_SIZE=1536 — model uses only ~11 GB VRAM at batch 768; bumping further
#   amortizes per-batch CPU overhead.
# DATA_WORKERS=8 — high-RAM Colab has 12 vCPUs; 8 leaves headroom for the
#   main process + background sync thread without saturating.
# Aux losses set to 0.0 — these regularizers come with heavy per-sample CPU
#   cost (saliency mask is a Python loop over 4096 cells per sample). The
#   train.py dataset now SKIPS the compute when the weight is 0, freeing
#   the DataLoader bottleneck. We can re-enable them in a follow-up run if
#   the BC-only val score plateaus.
# LOG_EVERY_BATCHES=50 — frequent logs so we see throughput + ETA early.
MAX_TRANS_PER_EP    = 1000
BATCH_SIZE          = 1536
DATA_WORKERS        = 8
LOG_EVERY_BATCHES   = 50
AUX_SALIENCY_WEIGHT = 0.0
AUX_RECON_WEIGHT    = 0.0
AUX_ARCHETYPE_WEIGHT = 0.0

print(f'run name : {FULL_RUN_NAME}')
print(f'data     : {LOCAL_INPUT}')
print(f'output   : {LOCAL_OUTPUT}')
print(f'will sync to: {DRIVE_OUTPUT} every 5 min')
print(f'max_transitions_per_episode: {MAX_TRANS_PER_EP}')
print(f'batch_size: {BATCH_SIZE}  data_workers: {DATA_WORKERS}  log_every: {LOG_EVERY_BATCHES}')
print(f'aux: saliency={AUX_SALIENCY_WEIGHT} recon={AUX_RECON_WEIGHT} archetype={AUX_ARCHETYPE_WEIGHT}')
print()

_stop_sync = threading.Event()

def _sync_loop():
    while not _stop_sync.is_set():
        if LOCAL_OUTPUT.exists():
            try:
                subprocess.run(
                    ['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{DRIVE_OUTPUT}/'],
                    check=False, capture_output=True,
                )
                print(f'[sync {time.strftime("%H:%M:%S")}] -> Drive', flush=True)
            except Exception as e:
                print(f'[sync ERR] {e}', flush=True)
        for _ in range(300):
            if _stop_sync.is_set():
                return
            time.sleep(1)

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('background sync started (5-min interval)')

training_succeeded = False
try:
    !cd {LOCAL_WORKDIR} && python -u -m src.train \
        --project-root {LOCAL_WORKDIR} \
        --data {LOCAL_INPUT} \
        --output-dir {LOCAL_OUTPUT} \
        --hardware-profile a100 \
        --batch-size {BATCH_SIZE} \
        --data-workers {DATA_WORKERS} \
        --max-transitions-per-episode {MAX_TRANS_PER_EP} \
        --log-every-batches {LOG_EVERY_BATCHES} \
        --aux-saliency-weight {AUX_SALIENCY_WEIGHT} \
        --aux-recon-weight {AUX_RECON_WEIGHT} \
        --aux-archetype-weight {AUX_ARCHETYPE_WEIGHT} \
        {epochs_arg} \
        --online-val-every 2 \
        2>&1 | tee {LOCAL_OUTPUT}/train.log
    if (LOCAL_OUTPUT / 'summary.json').exists():
        training_succeeded = True
finally:
    _stop_sync.set()
    print('background sync stopped')

print(f'final sync -> {DRIVE_OUTPUT}')
subprocess.run(['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{DRIVE_OUTPUT}/'], check=False)

if training_succeeded:
    (LOCAL_OUTPUT / 'training_complete.flag').write_text('ok\n')
    subprocess.run(['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{DRIVE_OUTPUT}/'], check=False)
    print('training_succeeded=True (summary.json present, marker written)')
else:
    print('training_succeeded=False (no summary.json — auto-exit will be skipped)')
print('done')

## 9. Auto-exit Colab on successful completion

If §8 finished cleanly (`summary.json` written + `training_complete.flag` present), this cell **disconnects the runtime** so you stop burning compute units.

The cell waits 60 seconds first so you can interrupt if you want to keep the runtime alive (e.g., to inspect outputs locally). After the wait, `runtime.unassign()` shuts down the GPU instance.

**If §8 failed or was interrupted**, this cell prints a warning and does NOT disconnect — you keep the runtime so you can debug.

Set `AUTO_EXIT = False` to disable this entirely.

In [ ]:
AUTO_EXIT = True
WAIT_SEC = 60

import time

flag = LOCAL_OUTPUT / 'training_complete.flag'
summary = LOCAL_OUTPUT / 'summary.json'

if not AUTO_EXIT:
    print('AUTO_EXIT=False — runtime kept alive')
elif not flag.exists():
    print(f'training_complete.flag NOT found at {flag}')
    print('§8 likely failed or was interrupted. Keeping runtime alive for debugging.')
elif not summary.exists():
    print(f'summary.json NOT found at {summary}')
    print('Training may not have finished cleanly. Keeping runtime alive for debugging.')
else:
    print(f'training completed cleanly:')
    print(f'  flag: {flag}')
    print(f'  summary: {summary}')
    print(f'  best.pth on Drive: {DRIVE_OUTPUT}/checkpoints/best.pth')
    print()
    print(f'Disconnecting runtime in {WAIT_SEC}s. Click Stop on this cell to abort.')
    for s in range(WAIT_SEC, 0, -10):
        print(f'  ...{s}s', flush=True)
        time.sleep(10)
    print('disconnecting now — this stops compute unit billing.')
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f'runtime.unassign() failed: {e}')
        print('You may need to stop the runtime manually via the Colab UI.')

## 10. Resuming after a disconnect

If Colab kicked you off mid-training, restart the runtime, re-run cells 1–6 (everything before training), then run this cell **instead of** §8.

The inline sync in §8 copies `last.pth` to Drive every 5 min, so the worst-case loss is 5 min of training. This cell finds the most recent matching run on Drive, pulls the checkpoint to local SSD, and resumes training (with the same inline sync).

In [ ]:
import threading
import time
import subprocess

prior_runs = sorted(DRIVE_OUTPUT_BASE.glob(f'{RUN_NAME}_*'), reverse=True)
print(f'found {len(prior_runs)} prior runs matching {RUN_NAME}_*')
for p in prior_runs[:5]:
    print(f'  {p.name}')

if not prior_runs:
    raise SystemExit('No prior runs found to resume from.')

RESUME_FROM_DRIVE = prior_runs[0]
ckpt_dir = RESUME_FROM_DRIVE / 'checkpoints'
candidates = ['interrupt.pth', 'last.pth', 'best.pth']
resume_ckpt = next((ckpt_dir / c for c in candidates if (ckpt_dir / c).exists()), None)
if resume_ckpt is None:
    raise SystemExit(f'No checkpoint found in {ckpt_dir}')
print(f'resuming from: {resume_ckpt}')

if not LOCAL_OUTPUT.exists():
    LOCAL_OUTPUT.mkdir(parents=True)
!rsync -a "{RESUME_FROM_DRIVE}/" "{LOCAL_OUTPUT}/"
local_ckpt = LOCAL_OUTPUT / 'checkpoints' / resume_ckpt.name

_stop_sync = threading.Event()

def _sync_loop():
    while not _stop_sync.is_set():
        if LOCAL_OUTPUT.exists():
            try:
                subprocess.run(['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{RESUME_FROM_DRIVE}/'],
                               check=False, capture_output=True)
                print(f'[sync {time.strftime("%H:%M:%S")}] -> Drive', flush=True)
            except Exception as e:
                print(f'[sync ERR] {e}', flush=True)
        for _ in range(300):
            if _stop_sync.is_set():
                return
            time.sleep(1)

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('background sync re-armed')

epochs_arg = f'--epochs {EPOCHS_OVERRIDE}' if EPOCHS_OVERRIDE else ''
try:
    !cd {LOCAL_WORKDIR} && python -u -m src.train \
        --project-root {LOCAL_WORKDIR} \
        --data {LOCAL_INPUT} \
        --output-dir {LOCAL_OUTPUT} \
        --hardware-profile a100 \
        {epochs_arg} \
        --online-val-every 2 \
        --resume {local_ckpt} \
        2>&1 | tee -a {LOCAL_OUTPUT}/train.log
finally:
    _stop_sync.set()

subprocess.run(['rsync', '-a', f'{LOCAL_OUTPUT}/', f'{RESUME_FROM_DRIVE}/'], check=False)
print(f'final sync -> {RESUME_FROM_DRIVE}')